### PASO 4 (PIPELINE COMPLETO): IMAGEN → SUDOKU RESUELTO

### ¿Qué hace este notebook?

Integra todo el trabajo anterior en un solo pipeline:

```
FOTO DEL SUDOKU
       ↓
YOLO detecta la cuadrícula (bounding box)
       ↓
Perspective warp → cuadrícula 450×450 px perfecta
       ↓
Segmentar en 81 celdas de 50×50 px
       ↓
CNN (modelo_digitos_v3) reconoce cada dígito
       ↓
Backtracking resuelve el sudoku
       ↓
Overlay con la solución sobre la imagen original
       ↓
Resultado listo para Streamlit
```

### Plan

```
CELDA 1 → Imports y carga de modelos
CELDA 2 → Todas las funciones auxiliares (warp, segmentación, CNN)
CELDA 3 → imagen_a_matriz(): YOLO + warp + CNN → matriz 9×9
CELDA 4 → resolver_sudoku(): backtracking
CELDA 5 → mostrar_resultado(): pipeline completo con overlay
CELDA 6 → Probar con imágenes de test
CELDA 7 → Guardar pipeline.py para Streamlit
```

#### CELDA 1 — Imports y carga de modelos

In [ ]:
# ============================================
# IMPORTS Y CARGA DE MODELOS
# ============================================

from google.colab import drive
drive.mount('/content/drive')

!pip install ultralytics tensorflow opencv-python-headless -q

import os
import cv2
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from ultralytics import YOLO

RUTA_SUDOKU = '/content/drive/MyDrive/sudoku/'
RUTA_YOLO   = RUTA_SUDOKU + 'modelo_yolo.pt'
RUTA_CNN    = RUTA_SUDOKU + 'modelo_digitos_v3.keras'   # modelo final con datos reales
RUTA_TEST   = RUTA_SUDOKU + 'images/test/'

print("📦 Cargando modelos...")
modelo_yolo = YOLO(RUTA_YOLO)
modelo_cnn  = keras.models.load_model(RUTA_CNN)
print(f"✅ YOLO cargado")
print(f"✅ CNN cargada: {RUTA_CNN}")
print(f"\n   GPU disponible: {len(tf.config.list_physical_devices('GPU')) > 0}")

#### CELDA 2 — Funciones auxiliares

Todas las funciones del Paso 2b en un solo bloque: warp, segmentación y detección de dígito.

In [ ]:
# ============================================
# FUNCIONES AUXILIARES (del Paso 2b)
# ============================================

def encontrar_esquinas(img_rgb, box_yolo):
    """Encuentra las 4 esquinas reales de la cuadrícula usando OpenCV."""
    x1, y1, x2, y2 = box_yolo
    margen = 20
    H, W   = img_rgb.shape[:2]
    rx1, ry1 = max(0, x1-margen), max(0, y1-margen)
    rx2, ry2 = min(W, x2+margen), min(H, y2+margen)
    recorte  = img_rgb[ry1:ry2, rx1:rx2]
    gray     = cv2.cvtColor(recorte, cv2.COLOR_RGB2GRAY)
    blur     = cv2.GaussianBlur(gray, (5, 5), 0)
    binaria  = cv2.adaptiveThreshold(blur, 255,
                   cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
                   cv2.THRESH_BINARY_INV, 11, 2)
    contornos, _ = cv2.findContours(binaria, cv2.RETR_EXTERNAL,
                                     cv2.CHAIN_APPROX_SIMPLE)
    if not contornos:
        return None
    contorno_mayor = max(contornos, key=cv2.contourArea)
    if cv2.contourArea(contorno_mayor) < 0.3 * (rx2-rx1) * (ry2-ry1):
        return None
    epsilon = 0.02 * cv2.arcLength(contorno_mayor, True)
    approx  = cv2.approxPolyDP(contorno_mayor, epsilon, True)
    intentos = 0
    while len(approx) != 4 and intentos < 10:
        epsilon *= 1.2
        approx   = cv2.approxPolyDP(contorno_mayor, epsilon, True)
        intentos += 1
    if len(approx) != 4:
        rect   = cv2.minAreaRect(contorno_mayor)
        approx = cv2.boxPoints(rect).reshape(-1, 1, 2).astype(int)
    puntos = approx.reshape(-1, 2).astype(np.float32)
    puntos[:, 0] += rx1
    puntos[:, 1] += ry1
    return puntos


def ordenar_esquinas(puntos):
    """Ordena 4 puntos: arriba-izq, arriba-der, abajo-der, abajo-izq."""
    puntos  = puntos.reshape(4, 2)
    ordered = np.zeros((4, 2), dtype=np.float32)
    sumas   = puntos.sum(axis=1)
    diffs   = np.diff(puntos, axis=1).flatten()
    ordered[0] = puntos[np.argmin(sumas)]
    ordered[2] = puntos[np.argmax(sumas)]
    ordered[1] = puntos[np.argmin(diffs)]
    ordered[3] = puntos[np.argmax(diffs)]
    return ordered


def warp_perspectiva(img_rgb, esquinas, tamaño=450):
    """Transforma la cuadrícula torcida en un cuadrado perfecto."""
    origen  = ordenar_esquinas(esquinas)
    destino = np.float32([[0,0],[tamaño,0],[tamaño,tamaño],[0,tamaño]])
    M       = cv2.getPerspectiveTransform(origen, destino)
    return cv2.warpPerspective(img_rgb, M, (tamaño, tamaño))


def segmentar_celdas(grid_warped, tamaño_celda=50, margen=4):
    """Divide el grid warped en 81 celdas limpias de 28×28."""
    gray   = cv2.cvtColor(grid_warped, cv2.COLOR_RGB2GRAY)
    celdas = []
    for i in range(9):
        for j in range(9):
            y1c = i * tamaño_celda
            y2c = (i + 1) * tamaño_celda
            x1c = j * tamaño_celda
            x2c = (j + 1) * tamaño_celda
            celda = gray[y1c:y2c, x1c:x2c]
            # Margen extra en borde izquierdo y superior para eliminar líneas
            margen_izq = margen + 3 if j == 0 else margen
            margen_arr = margen + 3 if i == 0 else margen
            celda_sin_bordes = celda[margen_arr:tamaño_celda-margen,
                                     margen_izq:tamaño_celda-margen]
            celda_28  = cv2.resize(celda_sin_bordes, (28, 28))
            celda_bin = cv2.adaptiveThreshold(
                celda_28, 255,
                cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
                cv2.THRESH_BINARY_INV, 11, 4)
            kernel    = np.ones((2, 2), np.uint8)
            celda_bin = cv2.morphologyEx(celda_bin, cv2.MORPH_OPEN, kernel)
            celdas.append(celda_bin.astype('float32') / 255.0)
    return celdas


def hay_digito(celda_norm, umbral_area=0.06):
    """
    Detecta si la celda contiene un dígito midiendo el área de píxeles blancos.
    Versión simplificada: con el warp las celdas ya están limpias,
    no necesitamos filtro de forma.
    """
    area_bruta = float(np.sum(celda_norm > 0.5) / (28 * 28))
    return area_bruta > umbral_area, area_bruta


print("✅ Funciones auxiliares definidas")

#### CELDA 3 — imagen_a_matriz()

Función principal: recibe una imagen y devuelve la matriz 9×9 con los dígitos detectados.

In [ ]:
# ============================================
# FUNCIÓN: imagen_a_matriz()
# ============================================

def imagen_a_matriz(ruta_imagen, modelo_yolo, modelo_cnn,
                    tamaño_grid=450, tamaño_celda=50, margen=4):
    """
    Pipeline completo: imagen → matriz 9×9 con dígitos detectados.

    Entrada:
        ruta_imagen: str, ruta a la imagen del sudoku
        modelo_yolo: modelo YOLO entrenado
        modelo_cnn:  modelo CNN (modelo_digitos_v3)

    Salida:
        matriz:      array 9×9 con dígitos (0 = vacío)
        grid_warped: imagen de la cuadrícula corregida (para el overlay)
        resumen:     texto con el resultado
    """

    # ── FASE 1: YOLO detecta la cuadrícula ──────────────────────────────
    img     = cv2.imread(ruta_imagen)
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    results = modelo_yolo(ruta_imagen, verbose=False)

    if len(results[0].boxes) == 0:
        return None, None, "❌ YOLO no detectó la cuadrícula"

    box  = results[0].boxes[0]
    conf = float(box.conf[0])
    if conf < 0.5:
        return None, None, f"❌ Confianza YOLO demasiado baja ({conf:.0%})"

    x1, y1, x2, y2 = map(int, box.xyxy[0].tolist())

    # ── FASE 2: Perspective warp ─────────────────────────────────────────
    esquinas = encontrar_esquinas(img_rgb, (x1, y1, x2, y2))
    if esquinas is not None:
        grid_warped = warp_perspectiva(img_rgb, esquinas, tamaño_grid)
    else:
        esq_fb      = np.float32([[x1,y1],[x2,y1],[x2,y2],[x1,y2]])
        grid_warped = warp_perspectiva(img_rgb, esq_fb, tamaño_grid)

    # ── FASE 3: Segmentar y reconocer dígitos ────────────────────────────
    celdas = segmentar_celdas(grid_warped, tamaño_celda, margen)
    matriz = np.zeros((9, 9), dtype=int)
    n_dig  = 0

    for idx, celda in enumerate(celdas):
        i, j = idx // 9, idx % 9
        tiene, _ = hay_digito(celda)
        if not tiene:
            continue

        # Pasar a la CNN
        entrada  = celda.reshape(1, 28, 28, 1)
        pred     = modelo_cnn.predict(entrada, verbose=0)[0]
        pred[0]  = 0.0          # en sudoku no existe el dígito 0
        pred     = pred / pred.sum()
        digito   = int(np.argmax(pred))
        confianza = float(pred[digito])

        if confianza >= 0.70:   # umbral de confianza
            matriz[i][j] = digito
            n_dig += 1

    resumen = f"✅ {n_dig} dígitos detectados"
    return matriz, grid_warped, resumen


print("✅ imagen_a_matriz() definida")

#### CELDA 4 — resolver_sudoku(): backtracking

In [ ]:
# ============================================
# FUNCIÓN: resolver_sudoku() — Backtracking (algoritmo de fuerza bruta inteligente)
# ============================================
'''
Busca la primera celda vacía (valor = 0).
Prueba números del 1 al 9.
Verifica si el número es válido en esa posición (fila, columna, subcuadrícula).
Si es válido, lo coloca y llama recursivamente a resolver_sudoku().
Si la recursión falla, deshace el número (vuelve a 0) y prueba el siguiente.
Si no encuentra ningún número válido, devuelve False (sin solución).
'''

def es_valido(tablero, fila, col, num):
    """Comprueba si num es válido en la posición (fila, col)."""
    if num in tablero[fila]:
        return False
    if num in [tablero[i][col] for i in range(9)]:
        return False
    fi = (fila // 3) * 3
    ci = (col  // 3) * 3
    for i in range(fi, fi + 3):
        for j in range(ci, ci + 3):
            if tablero[i][j] == num:
                return False
    return True


def resolver_sudoku(tablero):
    """
    Resuelve el sudoku con backtracking.
    Modifica el tablero directamente.
    Devuelve True si lo resolvió, False si no tiene solución.
    """
    for i in range(9):
        for j in range(9):
            if tablero[i][j] == 0:
                for num in range(1, 10):
                    if es_valido(tablero, i, j, num):
                        tablero[i][j] = num
                        if resolver_sudoku(tablero):
                            return True
                        tablero[i][j] = 0
                return False
    return True


print("✅ resolver_sudoku() definida")

#### CELDA 5 — mostrar_resultado(): pipeline completo con overlay

Dibuja los dígitos de la solución sobre la imagen original.
- Los dígitos que estaban en las pistas se muestran en blanco,
- Los que resolvió el backtracking en verde.

In [ ]:
# ============================================
# FUNCIÓN: mostrar_resultado()
# ============================================

def mostrar_resultado(ruta_imagen, modelo_yolo, modelo_cnn):
    """
    Pipeline completo: imagen → overlay con la solución.

    Salida:
        imagen_resultado: imagen con la solución dibujada encima
        matriz_original:  dígitos detectados por la CNN (0 = vacío)
        matriz_resuelta:  sudoku completo tras el backtracking
        mensaje:          texto con el resultado
    """

    # ── Detectar dígitos ─────────────────────────────────────────────────
    matriz, grid_warped, resumen = imagen_a_matriz(
        ruta_imagen, modelo_yolo, modelo_cnn
    )

    if matriz is None:
        return None, None, None, resumen

    n_pistas = int(np.sum(matriz > 0))

    if n_pistas < 17:
        msg = f"❌ Solo {n_pistas} pistas detectadas. Mínimo 17 para resolver."
        return None, matriz, None, msg

    # ── Resolver con backtracking ────────────────────────────────────────
    matriz_resuelta = matriz.copy()
    if not resolver_sudoku(matriz_resuelta):
        msg = "❌ El sudoku no tiene solución. Algún dígito fue mal leído."
        return None, matriz, None, msg

    # ── Dibujar overlay sobre el grid warped ─────────────────────────────
    # Usamos el grid warped (450×450) como lienzo
    overlay = grid_warped.copy()
    h, w    = overlay.shape[:2]
    cell_h  = h // 9
    cell_w  = w // 9

    for i in range(9):
        for j in range(9):
            if matriz_resuelta[i][j] == 0:
                continue

            # Centro de la celda
            cx = j * cell_w + cell_w // 2
            cy = i * cell_h + cell_h // 2

            # Dígitos originales (pistas) en blanco, solución en verde
            es_pista = matriz[i][j] != 0
            color    = (255, 255, 255) if es_pista else (0, 200, 0)

            cv2.putText(
                overlay,
                str(matriz_resuelta[i][j]),
                (cx - 8, cy + 10),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.8,      # tamaño de fuente
                color,
                2,        # grosor
                cv2.LINE_AA
            )

    msg = f"✅ Sudoku resuelto ({n_pistas} pistas detectadas)"
    return overlay, matriz, matriz_resuelta, msg


print("✅ mostrar_resultado() definida")

#### CELDA 6 — Probar el pipeline con imágenes de test

In [ ]:
# ============================================
# PROBAR EL PIPELINE COMPLETO
# ============================================

imagenes_test = [f for f in os.listdir(RUTA_TEST)
                 if f.endswith(('.jpg', '.png'))][:5]

print(f"🚀 Probando pipeline con {len(imagenes_test)} imágenes de test...")
print("="*60)

resueltos = 0

for nombre_img in imagenes_test:
    ruta = RUTA_TEST + nombre_img
    print(f"\n📷 {nombre_img}")

    overlay, matriz, solucion, mensaje = mostrar_resultado(
        ruta, modelo_yolo, modelo_cnn
    )

    print(f"   {mensaje}")

    if overlay is not None:
        resueltos += 1

        # Mostrar imagen original y resultado lado a lado
        img_orig = cv2.cvtColor(cv2.imread(ruta), cv2.COLOR_BGR2RGB)
        fig, axes = plt.subplots(1, 2, figsize=(12, 6))
        axes[0].imshow(img_orig)
        axes[0].set_title('Imagen original', fontsize=11)
        axes[0].axis('off')
        axes[1].imshow(overlay)
        axes[1].set_title('Solución (blanco=pista, verde=resuelto)', fontsize=11)
        axes[1].axis('off')
        plt.tight_layout()
        plt.show()

        # Imprimir matriz resuelta
        print("   ┌─────────┬─────────┬─────────┐")
        for i, fila in enumerate(solucion):
            if i > 0 and i % 3 == 0:
                print("   ├─────────┼─────────┼─────────┤")
            s = ""
            for j, n in enumerate(fila):
                if j % 3 == 0: s += "│ "
                s += str(int(n)) + " "
            print(f"   {s}│")
        print("   └─────────┴─────────┴─────────┘")

print(f"\n{'='*60}")
print(f"📊 Resueltos: {resueltos}/{len(imagenes_test)}")

#### CELDA 7 — Guardar pipeline.py para Streamlit

Exporta todas las funciones a un archivo `.py` que importará la app de Streamlit.

In [ ]:
# ============================================
# GUARDAR pipeline.py
# ============================================

RUTA_APP = RUTA_SUDOKU + 'app/'
os.makedirs(RUTA_APP, exist_ok=True)

pipeline_py = '''"""
pipeline.py — Pipeline completo para resolver sudokus desde fotos.
Importado por app.py (Streamlit).
"""

import cv2
import numpy as np
from ultralytics import YOLO
from tensorflow import keras


def cargar_modelos(ruta_yolo, ruta_cnn):
    modelo_yolo = YOLO(ruta_yolo)
    modelo_cnn  = keras.models.load_model(ruta_cnn)
    return modelo_yolo, modelo_cnn


def encontrar_esquinas(img_rgb, box_yolo):
    x1, y1, x2, y2 = box_yolo
    margen = 20
    H, W   = img_rgb.shape[:2]
    rx1, ry1 = max(0, x1-margen), max(0, y1-margen)
    rx2, ry2 = min(W, x2+margen), min(H, y2+margen)
    recorte  = img_rgb[ry1:ry2, rx1:rx2]
    gray     = cv2.cvtColor(recorte, cv2.COLOR_RGB2GRAY)
    blur     = cv2.GaussianBlur(gray, (5, 5), 0)
    binaria  = cv2.adaptiveThreshold(blur, 255,
                   cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
                   cv2.THRESH_BINARY_INV, 11, 2)
    contornos, _ = cv2.findContours(binaria, cv2.RETR_EXTERNAL,
                                     cv2.CHAIN_APPROX_SIMPLE)
    if not contornos:
        return None
    contorno_mayor = max(contornos, key=cv2.contourArea)
    if cv2.contourArea(contorno_mayor) < 0.3 * (rx2-rx1) * (ry2-ry1):
        return None
    epsilon = 0.02 * cv2.arcLength(contorno_mayor, True)
    approx  = cv2.approxPolyDP(contorno_mayor, epsilon, True)
    intentos = 0
    while len(approx) != 4 and intentos < 10:
        epsilon *= 1.2
        approx   = cv2.approxPolyDP(contorno_mayor, epsilon, True)
        intentos += 1
    if len(approx) != 4:
        rect   = cv2.minAreaRect(contorno_mayor)
        approx = cv2.boxPoints(rect).reshape(-1, 1, 2).astype(int)
    puntos = approx.reshape(-1, 2).astype(np.float32)
    puntos[:, 0] += rx1
    puntos[:, 1] += ry1
    return puntos


def ordenar_esquinas(puntos):
    puntos  = puntos.reshape(4, 2)
    ordered = np.zeros((4, 2), dtype=np.float32)
    sumas   = puntos.sum(axis=1)
    diffs   = np.diff(puntos, axis=1).flatten()
    ordered[0] = puntos[np.argmin(sumas)]
    ordered[2] = puntos[np.argmax(sumas)]
    ordered[1] = puntos[np.argmin(diffs)]
    ordered[3] = puntos[np.argmax(diffs)]
    return ordered


def warp_perspectiva(img_rgb, esquinas, tamaño=450):
    origen  = ordenar_esquinas(esquinas)
    destino = np.float32([[0,0],[tamaño,0],[tamaño,tamaño],[0,tamaño]])
    M       = cv2.getPerspectiveTransform(origen, destino)
    return cv2.warpPerspective(img_rgb, M, (tamaño, tamaño))


def segmentar_celdas(grid_warped, tamaño_celda=50, margen=4):
    gray   = cv2.cvtColor(grid_warped, cv2.COLOR_RGB2GRAY)
    celdas = []
    for i in range(9):
        for j in range(9):
            y1c = i * tamaño_celda
            y2c = (i + 1) * tamaño_celda
            x1c = j * tamaño_celda
            x2c = (j + 1) * tamaño_celda
            celda = gray[y1c:y2c, x1c:x2c]
            margen_izq = margen + 3 if j == 0 else margen
            margen_arr = margen + 3 if i == 0 else margen
            celda_sin_bordes = celda[margen_arr:tamaño_celda-margen,
                                     margen_izq:tamaño_celda-margen]
            celda_28  = cv2.resize(celda_sin_bordes, (28, 28))
            celda_bin = cv2.adaptiveThreshold(
                celda_28, 255,
                cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
                cv2.THRESH_BINARY_INV, 11, 4)
            kernel    = np.ones((2, 2), np.uint8)
            celda_bin = cv2.morphologyEx(celda_bin, cv2.MORPH_OPEN, kernel)
            celdas.append(celda_bin.astype("float32") / 255.0)
    return celdas


def hay_digito(celda_norm, umbral_area=0.06):
    area_bruta = float(np.sum(celda_norm > 0.5) / (28 * 28))
    return area_bruta > umbral_area, area_bruta


def es_valido(tablero, fila, col, num):
    if num in tablero[fila]: return False
    if num in [tablero[i][col] for i in range(9)]: return False
    fi = (fila // 3) * 3
    ci = (col  // 3) * 3
    for i in range(fi, fi + 3):
        for j in range(ci, ci + 3):
            if tablero[i][j] == num: return False
    return True


def resolver_sudoku(tablero):
    for i in range(9):
        for j in range(9):
            if tablero[i][j] == 0:
                for num in range(1, 10):
                    if es_valido(tablero, i, j, num):
                        tablero[i][j] = num
                        if resolver_sudoku(tablero): return True
                        tablero[i][j] = 0
                return False
    return True


def imagen_a_matriz(ruta_imagen, modelo_yolo, modelo_cnn):
    img     = cv2.imread(ruta_imagen)
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    results = modelo_yolo(ruta_imagen, verbose=False)
    if len(results[0].boxes) == 0:
        return None, None, "No se detectó la cuadrícula"
    box  = results[0].boxes[0]
    if float(box.conf[0]) < 0.5:
        return None, None, "Confianza YOLO demasiado baja"
    x1, y1, x2, y2 = map(int, box.xyxy[0].tolist())
    esquinas = encontrar_esquinas(img_rgb, (x1, y1, x2, y2))
    if esquinas is not None:
        grid_warped = warp_perspectiva(img_rgb, esquinas)
    else:
        esq_fb      = np.float32([[x1,y1],[x2,y1],[x2,y2],[x1,y2]])
        grid_warped = warp_perspectiva(img_rgb, esq_fb)
    celdas = segmentar_celdas(grid_warped)
    matriz = np.zeros((9, 9), dtype=int)
    for idx, celda in enumerate(celdas):
        i, j = idx // 9, idx % 9
        tiene, _ = hay_digito(celda)
        if not tiene: continue
        entrada  = celda.reshape(1, 28, 28, 1)
        pred     = modelo_cnn.predict(entrada, verbose=0)[0]
        pred[0]  = 0.0
        pred     = pred / pred.sum()
        digito   = int(np.argmax(pred))
        if float(pred[digito]) >= 0.70:
            matriz[i][j] = digito
    return matriz, grid_warped, f"{int(np.sum(matriz > 0))} dígitos detectados"


def mostrar_resultado(ruta_imagen, modelo_yolo, modelo_cnn):
    matriz, grid_warped, resumen = imagen_a_matriz(ruta_imagen, modelo_yolo, modelo_cnn)
    if matriz is None:
        return None, None, None, resumen
    if int(np.sum(matriz > 0)) < 17:
        return None, matriz, None, f"Solo {int(np.sum(matriz > 0))} pistas. Mínimo 17."
    matriz_resuelta = matriz.copy()
    if not resolver_sudoku(matriz_resuelta):
        return None, matriz, None, "Sin solución. Algún dígito mal leído."
    overlay = grid_warped.copy()
    h, w    = overlay.shape[:2]
    cell_h, cell_w = h // 9, w // 9
    for i in range(9):
        for j in range(9):
            if matriz_resuelta[i][j] == 0: continue
            cx    = j * cell_w + cell_w // 2
            cy    = i * cell_h + cell_h // 2
            color = (255, 255, 255) if matriz[i][j] != 0 else (0, 200, 0)
            cv2.putText(overlay, str(matriz_resuelta[i][j]),
                        (cx - 8, cy + 10),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.8, color, 2, cv2.LINE_AA)
    return overlay, matriz, matriz_resuelta, f"Resuelto ({int(np.sum(matriz > 0))} pistas)"
'''

with open(RUTA_APP + 'pipeline.py', 'w') as f:
    f.write(pipeline_py)

print(f"✅ pipeline.py guardado en: {RUTA_APP}")
print(f"   Tamaño: {os.path.getsize(RUTA_APP + 'pipeline.py') / 1024:.1f} KB")
print(f"\n📌 Este archivo lo importará app.py en el Paso 5 Streamlit")